# Set the limits of this microscope

The limits say how far the stage may travel, and which objectives and optical configurations the driver may choose. Every change the driver sends is checked against them first; one outside them is refused before anything moves. Whatever you set here, the driver never allows more stage travel than NIS-Elements itself reports.

Until limits are saved here, the driver uses cautious defaults: x and y within NIS's travel, and z only 100 µm either way from where the focus stood when it connected.

You need NIS-Elements running, with `start_bridge.mac` started in it (Macro > Run Macro From File).

Run the cells from the top.

In [ ]:
from zmart_drivers.nikon import nis_elements_ar7
from zmart_drivers.nikon.nis_elements_ar7.driver_setup.limits.config import (
    adopt_limits,
    build_payload,
)
from zmart_drivers.nikon.nis_elements_ar7.driver_setup.limits.measure_limits import (
    CornerRecorder,
)

scope = nis_elements_ar7.connect(
    microscope="nikon-microscope"
)  # this microscope's name
corners = CornerRecorder(scope.client)
print("NIS's own travel:", scope.state()["observed"]["nis_travel"])

## Record the safe x and y travel

Drive the stage with the joystick (or in NIS) to each corner of the area that is safe for the sample holder in use, and run that corner's cell. It only reads where the stage stands. To record a corner again, move the stage and run its cell again.

In [ ]:
corners.record(1)

In [ ]:
corners.record(2)

In [ ]:
corners.record(3)

In [ ]:
corners.record(4)

## Record the safe z travel

Focus down to the lowest position that is safe, with the objective still clear of the sample and its holder, and run the first cell. Then focus up to the highest position you want to allow, and run the second.

In [ ]:
corners.record_z("low")

In [ ]:
corners.record_z("high")

## Check and save

The ranges the corners span are shown. Leave `objective_position` and `optical_configuration` empty to allow every one, or list the ones the driver may choose, for example `objective_position=[1, 2]`. Saving keeps this notebook beside the limits, as the record of what was measured. The new limits govern from the next connection.

In [ ]:
LIMITS = build_payload({axis: spec["range"] for axis, spec in corners.limits().items()})
LIMITS

In [ ]:
published = adopt_limits(LIMITS)
published

In [ ]:
scope.close()